# gemma3_4b_unsloth — Azerbaijani PAWS

Zero-shot evaluation and QLoRA adaptation. Run cells in order on a CUDA GPU with Internet enabled. Data download from this GitHub repository; no Kaggle dataset attachment is needed.

Set `HF_TOKEN` in the environment, a local ignored `.env`, Kaggle Secrets, or Colab Secrets. Accept the checkpoint access terms on Hugging Face when required. Never paste credentials into a cell.

`AZSEMBENCH_REPO`, `AZSEMBENCH_REF`, `AZSEMBENCH_OUTPUT_DIR` and optional `AZSEMBENCH_DATA_SHA256` configure the dataset and output location. Use a fresh output directory for a new run. Saved historical metrics are under `results/`; outputs in this notebook are cleared.


### Gemma 3 / Unsloth compatibility

This version uses Unsloth's Gemma 3 4B 4-bit checkpoint to avoid the T4 numerical issues seen with vanilla Gemma 3 quantized loading. TRL configuration is filtered against the installed runtime API before training.


### PAWS inference

PAWS is evaluated by comparing the model's next-token logits for label tokens `0` and `1`; no free-form generation or parser is used.


In [ ]:
%pip -q install -U unsloth unsloth_zoo trl peft accelerate datasets scikit-learn scipy python-dotenv


## Repository data and private configuration

In [ ]:
import os
import hashlib
from pathlib import Path
from urllib.request import urlopen
from dotenv import load_dotenv

load_dotenv()
REPO = os.getenv("AZSEMBENCH_REPO", "MehribanAliyeva/azsembench")
REF = os.getenv("AZSEMBENCH_REF", "main")
OUTPUT_ROOT = Path(os.getenv("AZSEMBENCH_OUTPUT_DIR", "outputs")).expanduser()
DATA_DIR = OUTPUT_ROOT / "datasets"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_URL = f"https://raw.githubusercontent.com/{REPO}/{REF}/data/model_ready/core_model_ready.csv"
CSV_PATH = DATA_DIR / "core_model_ready.csv"
with urlopen(DATA_URL, timeout=120) as response:
    payload = response.read()
CSV_PATH.write_bytes(payload)
DATA_SHA256 = hashlib.sha256(payload).hexdigest()
expected_hash = os.getenv("AZSEMBENCH_DATA_SHA256")
if expected_hash and DATA_SHA256 != expected_hash:
    raise ValueError("Dataset hash differs from AZSEMBENCH_DATA_SHA256.")
print("Dataset:", DATA_URL)
print("SHA-256:", DATA_SHA256)

token = os.getenv("HF_TOKEN", "").strip()
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN") or ""
    except Exception:
        pass
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass
if token:
    os.environ["HF_TOKEN"] = token


In [ ]:
from unsloth import FastModel
import re,gc,json,shutil,warnings,inspect
from pathlib import Path
import numpy as np,pandas as pd,torch,trl,peft
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,matthews_corrcoef,confusion_matrix
from peft import PeftModel
from trl import SFTTrainer,SFTConfig

warnings.filterwarnings("ignore")
print("trl",trl.__version__)
print("peft",peft.__version__)
print("torch",torch.__version__)

if not torch.cuda.is_available(): raise RuntimeError("Enable GPU in Kaggle.")
GPU_NAME=torch.cuda.get_device_name(0)
CC_MAJOR,CC_MINOR=torch.cuda.get_device_capability()
SEED=42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_ID="unsloth/gemma-3-4b-it-unsloth-bnb-4bit"
RUN_DIR = OUTPUT_ROOT / "llm_paws_gemma"
ZERO_DIR=RUN_DIR/"zero_shot"
FT_DIR=RUN_DIR/"finetuned"
ADAPTER_DIR=RUN_DIR/"adapter"
for p in [RUN_DIR,ZERO_DIR,FT_DIR]: p.mkdir(parents=True,exist_ok=True)

print("GPU",GPU_NAME,"CC",f"{CC_MAJOR}.{CC_MINOR}")


In [ ]:
MAX_SEQ_LEN=256
MAX_NEW_TOKENS=2
TRAIN_SUBSET_N=15000
NUM_EPOCHS=1
LEARNING_RATE=2e-4
TRAIN_BATCH_SIZE=1
GRAD_ACCUM=4
INFER_BATCH_SIZE=16
LORA_R=16
LORA_ALPHA=16
LORA_DROPOUT=0.0
SAVE_EVERY_BATCHES=25

# A new output directory is required when the data or model configuration changes.
RUN_METADATA = {
    "model_id": MODEL_ID, "seed": SEED, "data_url": DATA_URL,
    "data_sha256": DATA_SHA256, "max_seq_len": MAX_SEQ_LEN,
    "epochs": NUM_EPOCHS, "learning_rate": LEARNING_RATE,
    "grad_accum": GRAD_ACCUM, "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
    "train_subset_n": globals().get("TRAIN_SUBSET_N"),
}
metadata_path = RUN_DIR / "run_metadata.json"
if metadata_path.exists():
    if json.loads(metadata_path.read_text()) != RUN_METADATA:
        raise RuntimeError("Output configuration changed; choose a fresh AZSEMBENCH_OUTPUT_DIR.")
metadata_path.write_text(json.dumps(RUN_METADATA, indent=2) + "\n")


In [ ]:
# Use the repository schema and Azerbaijani text explicitly.
raw = pd.read_csv(CSV_PATH, dtype={"source_id": str})
required = {"source_id", "task", "split", "sentence1_az", "sentence2_az", "label"}
if required - set(raw.columns):
    raise ValueError(f"Missing columns: {sorted(required - set(raw.columns))}")
raw = raw.loc[raw["task"].eq("paraphrase_detection")].copy()
if raw[["sentence1_az", "sentence2_az", "label", "source_id"]].isna().any().any():
    raise ValueError("PAWS examples must have IDs, two translations and a label.")
labels = pd.to_numeric(raw["label"], errors="raise")
if not labels.isin([0, 1]).all():
    raise ValueError("PAWS labels must be 0 or 1.")
df = pd.DataFrame({
    "id": raw["split"].astype(str) + ":" + raw["source_id"].astype(str),
    "sentence1": raw["sentence1_az"].astype(str),
    "sentence2": raw["sentence2_az"].astype(str),
    "label": labels.astype(int),
    "split": raw["split"].astype(str),
    "task": "paws",
}).reset_index(drop=True)
if df["id"].duplicated().any():
    raise ValueError("Duplicate PAWS IDs; refusing ambiguous resumable inference.")
full_train_df = df.loc[df["split"].eq("train")].reset_index(drop=True)
dev_df = df.loc[df["split"].eq("dev")].reset_index(drop=True)
test_df = df.loc[df["split"].eq("test")].reset_index(drop=True)
if any(part.empty for part in [full_train_df, dev_df, test_df]):
    raise ValueError("Expected non-empty train, dev and test partitions.")
if len(full_train_df) > TRAIN_SUBSET_N:
    train_df, _ = train_test_split(
        full_train_df, train_size=TRAIN_SUBSET_N,
        stratify=full_train_df["label"], random_state=SEED,
    )
    train_df = train_df.reset_index(drop=True)
else:
    train_df = full_train_df.copy()
print({"full_train": len(full_train_df), "used_train": len(train_df),
       "dev": len(dev_df), "test": len(test_df)})


In [ ]:
SYSTEM_PROMPT="Sən Azərbaycan dilində parafraz aşkarlayırsan. Cavabda yalnız 0 və ya 1 qaytar."

def user_prompt(s1,s2):
    return (
        "İki Azərbaycan dilində cümlə veriləcək.\n"
        "Əgər cümlələr eyni mənanı ifadə edirsə, yalnız 1 yaz.\n"
        "Əgər mənaları fərqlidirsə, yalnız 0 yaz.\n"
        "Heç bir izah yazma.\n\n"
        f"Cümlə 1: {s1}\nCümlə 2: {s2}\nCavab:"
    )

def make_messages(s1,s2,answer=None):
    m=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":user_prompt(s1,s2)}]
    if answer is not None: m.append({"role":"assistant","content":str(int(answer))})
    return m

def render_chat(tok,s1,s2,answer=None,generation=False):
    return tok.apply_chat_template(
        make_messages(s1,s2,answer),
        tokenize=False,
        add_generation_prompt=generation,
    )


In [ ]:
def load_base():
    model, tok = FastModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
        load_in_8bit=False,
        full_finetuning=False,
    )
    text_tokenizer = getattr(tok, "tokenizer", tok)
    text_tokenizer.padding_side = "left"
    if text_tokenizer.pad_token is None:
        text_tokenizer.pad_token = text_tokenizer.eos_token
    FastModel.for_inference(model)
    return model,tok


In [ ]:
def _encode_label(tok,text):
    if hasattr(tok,"encode"):
        return tok.encode(text,add_special_tokens=False)
    if hasattr(tok,"tokenizer") and hasattr(tok.tokenizer,"encode"):
        return tok.tokenizer.encode(text,add_special_tokens=False)
    raise RuntimeError("Could not access tokenizer.encode")

def get_label_token_ids(tok):
    ids0=_encode_label(tok,"0")
    ids1=_encode_label(tok,"1")
    print("Tokenization for 0:",ids0)
    print("Tokenization for 1:",ids1)
    if len(ids0)!=1 or len(ids1)!=1:
        raise RuntimeError("Expected labels 0 and 1 to each be one token.")
    return ids0[0],ids1[0]

def _tokenize_batch(tok,prompts,device):
    try:
        batch=tok(text=prompts,return_tensors="pt",padding=True,truncation=True,max_length=MAX_SEQ_LEN,add_special_tokens=False)
    except TypeError:
        batch=tok(prompts,return_tensors="pt",padding=True,truncation=True,max_length=MAX_SEQ_LEN,add_special_tokens=False)
    return {k:(v.to(device) if hasattr(v,"to") else v) for k,v in batch.items()}

@torch.inference_mode()
def classify_batch(model,tok,batch_df):
    prompts=[render_chat(tok,row.sentence1,row.sentence2,generation=True) for _,row in batch_df.iterrows()]
    inputs=_tokenize_batch(tok,prompts,model.device)
    outputs=model(**inputs,use_cache=False)
    next_logits=outputs.logits[:,-1,:]
    label_logits=torch.stack([next_logits[:,LABEL0_ID],next_logits[:,LABEL1_ID]],dim=-1)
    probs=torch.softmax(label_logits.float(),dim=-1)
    preds=torch.argmax(label_logits,dim=-1)
    return preds.detach().cpu().numpy().astype(int),probs.detach().cpu().numpy()


In [ ]:
def paws_metrics(df):
    d=df.dropna(subset=["prediction"]).copy()
    if len(d)==0:
        return {"n":len(df),"parsed_n":0,"parse_coverage":0.0,"accuracy":np.nan,"precision":np.nan,"recall":np.nan,"f1":np.nan,"mcc":np.nan}
    y=d.gold.astype(int).to_numpy(); p=d.prediction.astype(int).to_numpy()
    tn,fp,fn,tp=confusion_matrix(y,p,labels=[0,1]).ravel()
    return {
        "n":len(df),"parsed_n":len(d),"parse_coverage":len(d)/max(len(df),1),
        "accuracy":accuracy_score(y,p),
        "precision":precision_score(y,p,zero_division=0),
        "recall":recall_score(y,p,zero_division=0),
        "f1":f1_score(y,p,zero_division=0),
        "mcc":matthews_corrcoef(y,p),
        "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp),
    }


In [ ]:
def run_batched(model, tokenizer, source_df, out_path, condition, allow_resume):
    out_path = Path(out_path)

    if allow_resume and out_path.exists():
        old = pd.read_csv(out_path)
        done = set(old.id.astype(str))
    else:
        if out_path.exists():
            out_path.unlink()
        done = set()

    pending = source_df[
        ~source_df.id.astype(str).isin(done)
    ].reset_index(drop=True)

    for batch_num, start in enumerate(
        range(0, len(pending), INFER_BATCH_SIZE),
        start=1,
    ):
        batch = pending.iloc[
            start:start + INFER_BATCH_SIZE
        ].copy()

        preds, probs = classify_batch(
            model,
            tokenizer,
            batch,
        )

        rows = []

        for i, (_, row) in enumerate(batch.iterrows()):
            rows.append({
                "id": str(row.id),
                "sentence1": row.sentence1,
                "sentence2": row.sentence2,
                "gold": int(row.label),
                "prediction": int(preds[i]),
                "prob_0": float(probs[i, 0]),
                "prob_1": float(probs[i, 1]),
                "condition": condition,
                "model": "gemma3_4b_unsloth",
                "model_id": MODEL_ID,
                "task": "paws",
                "seed": SEED,
            })

        new = pd.DataFrame(rows)

        if out_path.exists():
            old = pd.read_csv(out_path)
            allp = pd.concat(
                [old, new],
                ignore_index=True,
            ).drop_duplicates(
                "id",
                keep="last",
            )
        else:
            allp = new

        allp.to_csv(out_path, index=False)

        if (
            batch_num % SAVE_EVERY_BATCHES == 0
            or start + INFER_BATCH_SIZE >= len(pending)
        ):
            print(condition, len(allp), "/", len(source_df))

    return pd.read_csv(out_path)


## Zero-shot

In [ ]:
base_model, tokenizer = load_base()
LABEL0_ID, LABEL1_ID = get_label_token_ids(tokenizer)

smoke = test_df.head(min(16, len(test_df))).copy()

smoke_preds, smoke_probs = classify_batch(
    base_model,
    tokenizer,
    smoke,
)

for i, (_, row) in enumerate(smoke.iterrows()):
    print(
        i,
        "PRED:", int(smoke_preds[i]),
        "P(0):", round(float(smoke_probs[i, 0]), 4),
        "P(1):", round(float(smoke_probs[i, 1]), 4),
        "GOLD:", int(row.label),
    )

print("Smoke test passed: every example received a 0/1 prediction.")

ZERO_PATH = ZERO_DIR / "predictions.csv"

zero_preds = run_batched(
    base_model,
    tokenizer,
    test_df,
    ZERO_PATH,
    "zero_shot",
    True,
)

zero_metrics = paws_metrics(zero_preds)

zero_metrics.update({
    "model": "gemma3_4b_unsloth",
    "model_id": MODEL_ID,
    "task": "paws",
    "condition": "zero_shot",
    "seed": SEED,
})

with open(ZERO_DIR / "metrics.json", "w") as f:
    json.dump(
        zero_metrics,
        f,
        indent=2,
        allow_nan=True,
    )

print(zero_metrics)


## Build training dataset

In [ ]:
def build_prompt(r):
    return render_chat(tokenizer,r.sentence1,r.sentence2,generation=True)

train_ds=Dataset.from_dict({
    "prompt":[build_prompt(r) for _,r in train_df.iterrows()],
    "completion":[str(int(r.label)) for _,r in train_df.iterrows()]
})
print(train_ds)


## Runtime-safe QLoRA training

In [ ]:
FastModel.for_training(base_model)

train_model=FastModel.get_peft_model(
    base_model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

sft_kwargs={
    "output_dir":str(RUN_DIR/"trainer"),
    "num_train_epochs":NUM_EPOCHS,
    "per_device_train_batch_size":TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps":GRAD_ACCUM,
    "learning_rate":LEARNING_RATE,
    "lr_scheduler_type":"cosine",
    "warmup_steps":50,
    "logging_steps":50,
    "eval_strategy":"no",
    "save_strategy":"no",
    "optim":"adamw_8bit",
    "fp16":True,
    "bf16":False,
    "max_length":MAX_SEQ_LEN,
    "packing":True,
    "completion_only_loss":True,
    "report_to":"none",
    "seed":SEED,
    "data_seed":SEED,
}

supported=set(inspect.signature(SFTConfig).parameters.keys())
unsupported=sorted(k for k in sft_kwargs if k not in supported)
if unsupported: print("Omitting unsupported SFTConfig args:",unsupported)
args=SFTConfig(**{k:v for k,v in sft_kwargs.items() if k in supported})

trainer_sig=set(inspect.signature(SFTTrainer).parameters.keys())
trainer_kwargs={"model":train_model,"args":args,"train_dataset":train_ds}
if "processing_class" in trainer_sig: trainer_kwargs["processing_class"]=tokenizer
elif "tokenizer" in trainer_sig: trainer_kwargs["tokenizer"]=tokenizer

trainer=SFTTrainer(**trainer_kwargs)
trainer.train()
trainer.model.save_pretrained(ADAPTER_DIR)
try: tokenizer.save_pretrained(ADAPTER_DIR)
except Exception: pass
pd.DataFrame(trainer.state.log_history).to_csv(RUN_DIR/"trainer_log.csv",index=False)
print("Training finished; adapter saved to",ADAPTER_DIR)


## Reload fresh base + adapter

In [ ]:
del trainer,train_model,base_model
gc.collect()
torch.cuda.empty_cache()

fresh_base,fresh_tokenizer=load_base()
LABEL0_ID,LABEL1_ID=get_label_token_ids(fresh_tokenizer)
finetuned_model=PeftModel.from_pretrained(fresh_base,str(ADAPTER_DIR),is_trainable=False)
finetuned_model.eval()
loaded=[n for n,_ in finetuned_model.named_parameters() if "lora_" in n]
print("Reloaded LoRA tensors:",len(loaded))
print("Active adapter:",getattr(finetuned_model,"active_adapter",None))
if not loaded: raise RuntimeError("Adapter did not reload")


## Fine-tuned test

In [ ]:
FT_PATH=FT_DIR/"predictions.csv"
if FT_PATH.exists(): FT_PATH.unlink()

fine_preds=run_batched(finetuned_model,fresh_tokenizer,test_df,FT_PATH,"qlora_finetuned",False)
fine_metrics=paws_metrics(fine_preds)
fine_metrics.update({"model":"gemma3_4b_unsloth","model_id":MODEL_ID,"task":"paws","condition":"qlora_finetuned","seed":SEED})
with open(FT_DIR/"metrics.json","w") as f: json.dump(fine_metrics,f,indent=2,allow_nan=True)
print(fine_metrics)


## Verify + save

In [ ]:
z=zero_preds.sort_values("id").reset_index(drop=True)
f=fine_preds.sort_values("id").reset_index(drop=True)
if not np.array_equal(z.id.astype(str).to_numpy(),f.id.astype(str).to_numpy()):
    raise RuntimeError("Prediction IDs do not align")

same=((pd.isna(z.prediction)&pd.isna(f.prediction))|(z.prediction==f.prediction)).to_numpy()
n_same=int(same.sum()); n_diff=int((~same).sum())

print("Total examples:",len(z))
print("Same predictions:",n_same)
print("Different predictions:",n_diff)
print("Fraction changed:",n_diff/len(z))

comparison=pd.DataFrame([zero_metrics,fine_metrics])
comparison.to_csv(RUN_DIR/"all_metrics.csv",index=False)
display(comparison)

run_config={
    "model_id":MODEL_ID,
    "task":"paws",
    "seed":SEED,
    "train_subset_n":len(train_df),
    "full_train_n":len(full_train_df),
    "dev_n":len(dev_df),
    "test_n":len(test_df),
    "epochs":NUM_EPOCHS,
    "max_seq_len":MAX_SEQ_LEN,
        "infer_batch_size":INFER_BATCH_SIZE,
    "packing":True,
    "learning_rate":LEARNING_RATE,
    "grad_accum":GRAD_ACCUM,
}
with open(RUN_DIR/"run_config.json","w") as f: json.dump(run_config,f,indent=2)

print("Results and adapter saved under:", RUN_DIR.resolve())
